In [1]:
import pandas as pd
import numpy as np

In [2]:
missing_values=['?','-',' ','NA','N/A','NaN']
df=pd.read_csv("abc_rest1.csv", na_values=missing_values)

In [3]:
df

,cust_id,cust_name,room_rating,food_rating,service_rating,cleanliness_rating,overall_feedback
0,1,AB,1.0,2.0,2,2,BAD
1,2,CD,1.0,3.0,3,3,AVERAGE
2,3,EF,3.0,4.0,4,4,GOOD
3,4,GH,2.0,5.0,5,5,GOOD
4,5,IJ,1.0,1.0,1,1,BAD
5,6,KL,NaN,2.0,3,3,AVERAGE
6,7,MN,NaN,NaN,4,4,GOOD
7,8,OP,NaN,4.0,4,5,GOOD
8,9,QR,NaN,NaN,1,1,BAD
9,10,ST,NaN,5.0,5,5,GOOD


In [4]:
df.isnull().sum()

cust_id               0
cust_name             0
room_rating           5
food_rating           2
service_rating        0
cleanliness_rating    0
overall_feedback      0
dtype: int64

In [5]:
df.dtypes

cust_id                 int64
cust_name              object
room_rating           float64
food_rating           float64
service_rating          int64
cleanliness_rating      int64
overall_feedback       object
dtype: object

In [6]:
fill_values={"room_rating":df['room_rating'].mean(),
            "food_rating":df['food_rating'].mean()}

In [7]:
fill_values

{'room_rating': 1.6, 'food_rating': 3.25}

In [8]:
df.fillna(fill_values, inplace = True)

In [9]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [10]:
#cols_to_encd= ['cust_name', 'overall_feedback']

In [11]:
df["cust_name"] = le.fit_transform(df["cust_name"])

In [12]:
df["overall_feedback"] = le.fit_transform(df["overall_feedback"])

In [13]:
# Seperating predictor and class columns
x = df.iloc[:,2:6]
y = df["overall_feedback"]

In [14]:
x

,room_rating,food_rating,service_rating,cleanliness_rating
0,1.0,2.00,2,2
1,1.0,3.00,3,3
2,3.0,4.00,4,4
3,2.0,5.00,5,5
4,1.0,1.00,1,1
5,1.6,2.00,3,3
6,1.6,3.25,4,4
7,1.6,4.00,4,5
8,1.6,3.25,1,1
9,1.6,5.00,5,5


In [15]:
y

0    1
1    0
2    2
3    2
4    1
5    0
6    2
7    2
8    1
9    2
Name: overall_feedback, dtype: int32

In [16]:
# Now let's do scaling
from sklearn.preprocessing import RobustScaler
rs = RobustScaler()

In [17]:
x = rs.fit_transform(x)
print(x)

[[-1.33333333 -0.71428571 -0.85714286 -0.6       ]
 [-1.33333333 -0.14285714 -0.28571429 -0.2       ]
 [ 3.11111111  0.42857143  0.28571429  0.2       ]
 [ 0.88888889  1.          0.85714286  0.6       ]
 [-1.33333333 -1.28571429 -1.42857143 -1.        ]
 [ 0.         -0.71428571 -0.28571429 -0.2       ]
 [ 0.          0.          0.28571429  0.2       ]
 [ 0.          0.42857143  0.28571429  0.6       ]
 [ 0.          0.         -1.42857143 -1.        ]
 [ 0.          1.          0.85714286  0.6       ]]


In [18]:
# Now let's do Feature Selection
from sklearn.decomposition import PCA
pca = PCA(n_components=2)

In [19]:
x_pca = pca.fit_transform(x)

In [20]:
print('Shape before PCA: ',x.shape)
print('Shape before PCA: ',x_pca.shape)

Shape before PCA:  (10, 4)
Shape before PCA:  (10, 2)


In [21]:
# Now let's split the features into training and test data into 80:20 respectively.
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x_pca, y, test_size = 0.2)

In [22]:
x_train

array([[-1.73831682, -0.04740481],
       [ 0.28462315, -0.35981323],
       [ 0.57595729, -0.67133611],
       [-1.14494532, -0.71271162],
       [ 1.04448496, -1.17127169],
       [ 2.79071711,  1.52882673],
       [-0.82953011,  1.05150059],
       [-0.36424512,  0.3542108 ]])

In [23]:
x_test

array([[-2.33168832,  0.617902  ],
       [ 1.71294315, -0.58990266]])

In [24]:
# Now let's go for ensembling modelusing voting
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score


In [25]:
# Create individual models
knn = KNeighborsClassifier()
dt = DecisionTreeClassifier(criterion= 'entropy')
svm = SVC()


In [26]:
# Create an ensemble model using VotingClassifier
ensemble_model = VotingClassifier(estimators = [
    ("K Nearest Neighbor", knn),
    ("Decision Tree", dt),
    ("Standard Vector Machine", svm)
], voting = "hard")

In [27]:
# Train the ensembled model
ensemble_model.fit(x_train, y_train)

VotingClassifier(estimators=[('K Nearest Neighbor', KNeighborsClassifier()),
                             ('Decision Tree',
                              DecisionTreeClassifier(criterion='entropy')),
                             ('Standard Vector Machine', SVC())])

In [28]:
prediction = ensemble_model.predict(x_test)

In [29]:
prediction

array([1, 2])

In [30]:
accuracy = accuracy_score(prediction, y_test)
print(f"Ensembled model's accuracy: {accuracy}")

Ensembled model's accuracy: 1.0


# Let's go with the individual models

In [31]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
rf = RandomForestClassifier()
nb = GaussianNB()

In [32]:
models = {
    "Decision tree": dt,
    "K-Nearest Neighbor": knn,
    "Random Forest": rf,
    "Naive Bayes": nb,
    "Standard Vector Machine": svm
}

In [33]:
print(type(models))

<class 'dict'>


In [34]:
# Initializing variables to track the highest accuracy and the corresponding model
highest_accuracy = 0
best_model = None
model_name = None

In [35]:
# Using for loop to taing each model individually
for name, model in models.items():
    model.fit(x_train, y_train)
   # Make prediction
    pred = model.predict(x_test)
   # print(y_test,"\t",pred,"\n")
    
   # Calculate accuracy
    accuracy1 = accuracy_score(pred, y_test)
    print(f"Model name: {name}, Accuracy: {accuracy}")
    if accuracy1 > highest_accuracy:
        highest_accuracy = accuracy1
        best_model = model
        model_name = name
print(f"\n{highest_accuracy}")
print(model_name)
print(best_model)

Model name: Decision tree, Accuracy: 1.0
Model name: K-Nearest Neighbor, Accuracy: 1.0
Model name: Random Forest, Accuracy: 1.0
Model name: Naive Bayes, Accuracy: 1.0
Model name: Standard Vector Machine, Accuracy: 1.0

1.0
Decision tree
DecisionTreeClassifier(criterion='entropy')
